<a href="https://colab.research.google.com/github/alex-degarate/DAnalytics/blob/main/taller_visu/taller_visual2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================================
# ETL PIPELINE: Superstore → Google Sheets (modelo Star)
# Visualización: Looker Studio
# ======================================================

# 1. Importamos librerías necesarias

In [ ]:
# --- 1. Librerías necesarias ---
import pandas as pd
import numpy as np
from datetime import datetime
from google.colab import auth
import gspread
from google.auth import default
from googleapiclient.discovery import build

# 2. Autenticación con Google

Usamos la librería google.auth para tener acceso a crear Documento de Google Sheets. Cada tab/hoja de cálculo, equivale a una tabla de una Base de Datos

In [ ]:
# --- 2. Autenticación con Google ---
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ Autenticado con Google correctamente.")

# 3. Importar / Cargar Dataset en formato csv

Vamos a user el dataset Superstore
<BR>
Ver en [Kaggle](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

Como venimos haciendo hasta ahora, descargamos el Dataset "Superstore" y lo guardamos en nuestro Google Drive. Luego lo cargamos en memoria para el procesamiento.

In [ ]:
import kagglehub

# Download latest version
#path = kagglehub.dataset_download("vivek468/superstore-dataset-final")

#print("Path to dataset files:", path)

In [ ]:
#!ls /root/.cache/kagglehub/datasets/vivek468/superstore-dataset-final/versions/1

In [ ]:
# Montar la unidad
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
# En caso que necesiten hace un unmount de la unidad
# drive.flush_and_unmount

In [ ]:
# Ahora copiar el dataset descargado a nuestro Google Drive
#import os, shutil
# Carpeta destino
#dest_dir = "/content/drive/MyDrive/datasets/"

# Copiar todos los archivos del dataset al destino
#for file in os.listdir(path):
#    shutil.copy(os.path.join(path, file), dest_dir)

# print("Archivos copiados a:", dest_dir)
# print("Contenido:", os.listdir(dest_dir))

#os.listdir(dest_dir)

In [ ]:
url = "https://github.com/alex-degarate/DAnalytics/raw/refs/heads/main/datasets/"
df = pd.read_csv(url + "superstore.csv", encoding="latin1")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [ ]:
df.sample()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
9041,9042,CA-2014-114335,9/28/2014,10/3/2014,Standard Class,XP-21865,Xylona Preis,Consumer,United States,Hollywood,...,33021,South,FUR-FU-10000277,Furniture,Furnishings,Deflect-o DuraMat Antistatic Studded Beveled M...,337.088,4,0.2,16.8544


# 4. Limpieza de datos

Asi como trabajamos en nuestra Pre-entrega, realizamos una limpieza de duplicados y nulos para alimentar con datos de calidad la próxima etapa.

In [ ]:
# Normalizar nombres de columnas
df.columns = df.columns.str.strip().str.replace(" ", "_").str.lower()

In [ ]:
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub-category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')

In [ ]:
# --- 4. Limpieza básica ---
df = df.dropna(subset=["order_date", "sales", "profit"])
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

In [ ]:
# Crear campos derivados
df["profit_margin"] = df["profit"] / df["sales"]
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_quarter"] = df["order_date"].dt.quarter
df["order_day"] = df["order_date"].dt.day
df["order_week"] = df["order_date"].dt.isocalendar().week.astype(int)

print("🧹 Limpieza y enriquecimiento de datos completado.")

🧹 Limpieza y enriquecimiento de datos completado.


# 5. Construcción de las dimensiones (modelo estrella)

Este es un concepto nuevo. Para poder representar los datos en Looker Studio (u otra herramienta de visualización), necesitamos que nuestros datos se encuentren organizados en una Base de Datos Operacional (OLAP). Esta base consta de una tabla de "facts" o "hechos" con métricas que luego se puede agregar (sumar, promediar, obtener máximos, etc) y tantan tablas de dimensiones como necesitemos. Estas tablas de dimensiones nos permiten analizar los datos desde múltiples perspectivas (como si fuera un cubo, donde cada lado se corresponde con una dimensión). Las dimensiones más comunes son: la tempora y la geográfica, en nuestro caso sumamos además la de categorías y de clientes.

A continuación cremos primeros las tablas correspondientes a cada dimensión. En nuestro caso usaremos tablas "desnomarmalizadas", y cada una tendra una clave primaria, que será referenciada desde la tabla de hechos, o fact table, que veremos debajo.

In [ ]:
# Dimensión Fecha
dim_date = df[["order_date", "order_year", "order_quarter", "order_month", "order_week", "order_day"]] \
    .drop_duplicates().reset_index(drop=True)
dim_date.insert(0, "date_id", range(1, len(dim_date)+1))

In [ ]:
# Dimensión Producto
dim_product = df[["category", "sub-category", "product_name"]] \
    .drop_duplicates().reset_index(drop=True)
dim_product.insert(0, "product_id", range(1, len(dim_product)+1))

In [ ]:
# Dimensión Geografía
dim_geo = df[["country", "region", "state", "city"]] \
    .drop_duplicates().reset_index(drop=True)
dim_geo.insert(0, "geo_id", range(1, len(dim_geo)+1))

In [ ]:
# Dimensión Cliente
dim_customer = df[["customer_id", "customer_name", "segment"]] \
    .drop_duplicates().reset_index(drop=True)
dim_customer.insert(0, "customer_sk", range(1, len(dim_customer)+1))

print("📚 Dimensiones creadas:")
print(f"  dim_date={len(dim_date)}, dim_product={len(dim_product)}, dim_geo={len(dim_geo)}, dim_customer={len(dim_customer)}")

📚 Dimensiones creadas:
  dim_date=1237, dim_product=1850, dim_geo=604, dim_customer=793


In [ ]:
display(dim_customer)

,customer_sk,customer_id,customer_name,segment
0,1,CG-12520,Claire Gute,Consumer
1,2,DV-13045,Darrin Van Huff,Corporate
2,3,SO-20335,Sean O'Donnell,Consumer
3,4,BH-11710,Brosina Hoffman,Consumer
4,5,AA-10480,Andrew Allen,Consumer
...,...,...,...,...
788,789,CJ-11875,Carl Jackson,Corporate
789,790,RS-19870,Roy Skaria,Home Office
790,791,SC-20845,Sung Chung,Consumer
791,792,RE-19405,Ricardo Emerson,Consumer


In [ ]:
!pip install odfpy
#!pip list | grep odfpy
#!pip show odfpy
#!ls -F /usr/local/lib/python3.12/dist-packages/odf

#import defusedxml
#from odfpy import defusedxml
from odf import office

anim.py		   elementtypes.py  number.py	     style.py
attrconverters.py  form.py	    odf2moinmoin.py  svg.py
chart.py	   grammar.py	    odf2xhtml.py     table.py
config.py	   __init__.py	    odfmanifest.py   teletype.py
dc.py		   load.py	    office.py	     text.py
dr3d.py		   manifest.py	    opendocument.py  thumbnail.py
draw.py		   math.py	    presentation.py  userfield.py
easyliststyle.py   meta.py	    __pycache__/     xforms.py
element.py	   namespaces.py    script.py


In [ ]:
# Use a shell command to list the contents of the odfpy installation directory
# !ls -F /usr/local/lib/python3.12/dist-packages

This command will show you the files and subdirectories within the `odfpy` package folder. If you see a list of files, it means the package is physically installed there.

In [ ]:
'''
# Make sure odfpy is imported. la fila inferior es la correcta !!!
# from odf import office  # OK !!!
df_to_save = dim_customer.copy()

df_to_save.to_excel("dim_customer.ods", engine="odf", index=False)
'''

In [ ]:
# Make sure odfpy is imported.
from odf import office

# List of DataFrame objects to save
# We need to get the actual DataFrame objects, not just their names as strings
dataframes_to_save = {
    "dim_date": dim_date,
    "dim_product": dim_product,
    "dim_geo": dim_geo,
    "dim_customer": dim_customer
}

for name, df_obj in dataframes_to_save.items():
    outfile = name + ".ods"
    # Save the DataFrame to an ODS file using the 'odf' engine
    df_obj.to_excel(outfile, engine="odf", index=False)

    print(f"✅ DataFrame '{name}' saved to '{outfile}' successfully!")

### Explanation:
*   **`import odfpy`**: This line ensures the `odfpy` library is available in your current Python environment.
*   **`df_to_save = dim_customer.copy()`**: It's good practice to work with a copy if you don't want to accidentally modify the original DataFrame.
*   **`df_to_save.to_excel("dim_customer.ods", engine="odf", index=False)`**: This is the crucial part. When saving a DataFrame to an OpenDocument Spreadsheet (`.ods`) file using `pandas.to_excel()`, you must explicitly set `engine="odf"`. The `index=False` prevents `pandas` from writing the DataFrame index as a column in the ODS file.

# 6. Construcción de la Tabla de Hechos (o bien conocida como Fact Table)

Esta es la tabla que mencionamos arteriormente, que solo contiene las métricas y las Claves Primarias a las tablas de dimensiones.

In [ ]:
# Mapear claves surrogate
date_map = dict(zip(dim_date["order_date"], dim_date["date_id"]))
prod_map = dict(zip(dim_product["product_name"], dim_product["product_id"]))
geo_map = dict(zip(dim_geo["city"], dim_geo["geo_id"]))
cust_map = dict(zip(dim_customer["customer_id"], dim_customer["customer_sk"]))

fact_sales = pd.DataFrame({
    "order_id": df["order_id"],
    "date_id": df["order_date"].map(date_map),
    "product_id": df["product_name"].map(prod_map),
    "geo_id": df["city"].map(geo_map),
    "customer_id": df["customer_id"].map(cust_map),
    "sales": df["sales"],
    "profit": df["profit"],
    "quantity": df["quantity"],
    "discount": df["discount"],
    "profit_margin": df["profit_margin"]
})

print(f"🧾 Tabla de hechos creada: {len(fact_sales)} registros.")

🧾 Tabla de hechos creada: 9994 registros.


In [ ]:
fact_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       9994 non-null   object 
 1   date_id        9994 non-null   int64  
 2   product_id     9994 non-null   int64  
 3   geo_id         9994 non-null   int64  
 4   customer_id    9994 non-null   int64  
 5   sales          9994 non-null   float64
 6   profit         9994 non-null   float64
 7   quantity       9994 non-null   int64  
 8   discount       9994 non-null   float64
 9   profit_margin  9994 non-null   float64
dtypes: float64(4), int64(5), object(1)
memory usage: 780.9+ KB


In [ ]:
fact_sales.head()

,order_id,date_id,product_id,geo_id,customer_id,sales,profit,quantity,discount,profit_margin
0,CA-2016-152156,1,1,294,1,261.9600,41.9136,2,0.00,0.1600
1,CA-2016-152156,1,2,294,1,731.9400,219.5820,3,0.00,0.3000
2,CA-2016-138688,2,3,2,2,14.6200,6.8714,2,0.00,0.4700
3,US-2015-108966,3,4,3,3,957.5775,-383.0310,5,0.45,-0.4000
4,US-2015-108966,3,5,3,3,22.3680,2.5164,2,0.20,0.1125


In [ ]:

# Make sure odfpy is imported. la fila inferior es la correcta !!!
# from odf import office  # OK !!!
#df_to_save = fact_sales.copy()

#df_to_save.to_excel("fact_sales.ods", engine="odf", index=False)


# 7. Poblar las tablas de un Documento de Google Sheets

Procedemos ahora a "poblar" nuestras tablas. En este caso usamos hojas de cálculo de Google Sheets, que luego podemos reemplazar por una Base de Datos Relacional, como ser PostgreSQL.

In [ ]:
# Buscamos el archivo "DW_Superstore_OLAP_Star"
# Si existe lo reutilizamos, sino lo creamos.
# La idea es que el ETL sea reproducible

# ----------------------------------------------------------
# Buscar o crear el archivo en Google Drive
service = build('drive', 'v3', credentials=creds)

file_name = "DW_Superstore_OLAP"
results = service.files().list(
    q=f"name='{file_name}' and mimeType='application/vnd.google-apps.spreadsheet'",
    spaces='drive'
).execute()

if results['files']:
    spreadsheet_id = results['files'][0]['id']
    spreadsheet = gc.open_by_key(spreadsheet_id)
    print("📂 Archivo existente encontrado:", spreadsheet.url)
else:
    spreadsheet = gc.create(file_name)
    print("🆕 Archivo nuevo creado:", spreadsheet.url)

# ----------------------------------------------------------
# Mantener "Hoja 1" intacta y limpiar las demás hojas antes de reescribirlas
worksheets = spreadsheet.worksheets()
existing_titles = [ws.title for ws in worksheets]

print("📄 Hojas existentes:", existing_titles)

# ----------------------------------------------------------
# Definir las tablas OLAP a subir
tables = {
    "dim_date": dim_date,
    "dim_product": dim_product,
    "dim_geo": dim_geo,
    "dim_customer": dim_customer,
    "fact_sales": fact_sales
}

# ----------------------------------------------------------
# Subir las tablas sin eliminar hojas (solo limpiar contenido)
for name, df_table in tables.items():
    # Convertir columnas datetime a string
    for col in df_table.columns:
        if np.issubdtype(df_table[col].dtype, np.datetime64):
            df_table[col] = df_table[col].dt.strftime('%Y-%m-%d')

    # Si la hoja ya existe, limpiar su contenido; si no, crearla
    existing_titles = [ws.title for ws in spreadsheet.worksheets()]
    if name in existing_titles:
        ws = spreadsheet.worksheet(name)
        ws.clear()
        print(f"🧹 Hoja existente '{name}' limpiada.")
    else:
        ws = spreadsheet.add_worksheet(
            title=name,
            rows=str(len(df_table) + 1),
            cols=str(len(df_table.columns) + 1)
        )
        print(f"🆕 Hoja '{name}' creada.")

    # Subir los datos (encabezados + valores)
    ws.update([df_table.columns.values.tolist()] + df_table.astype(str).values.tolist())
    print(f"📈 Hoja '{name}' actualizada correctamente ({len(df_table)} filas).")

print("✅ Data Warehouse OLAP actualizado correctamente.")
print("📊 URL:", spreadsheet.url)



Comprobar que las hojas del Google Sheet se completaron y continuar el armado del Dashboard desde Looker Studio.
<BR>
Como primer paso, se importan las tablas. Luego, a partir de los KPIs (primary key indicators) se arman los join y luego se agregan los gráficos y se configuran en el panel de propiedades.

# Validaciones

Aquí podemos hacer join/merge y agregaciones para validar los datos que arroja Looker Studio.

In [ ]:
fact_sales["customer_id"]=fact_sales["customer_id"].astype(str)
dim_customer["customer_sk"]=dim_customer["customer_sk"].astype(str)
facts_customer = pd.merge(fact_sales, dim_customer, left_on="customer_id", right_on="customer_sk", how="left")

In [ ]:
facts_customer.head()

In [ ]:
facts_customer.columns

In [ ]:
facts_customer.groupby("customer_name").agg(
    {"sales":"sum",
     "profit": "sum"}
)